In [1]:
import os
os.chdir('../')

In [16]:
import numpy as np
import torch
from pathlib import Path
from PIL import Image
from utils.fid import FIDInception
from tqdm import tqdm
import torchvision.transforms as T
from torchvision.transforms import InterpolationMode as IM

TARGET = 512  # 생성 해상도에 맞춰 256/512 등으로 설정
preproc = T.Compose([
    T.Resize(TARGET, interpolation=IM.BICUBIC, antialias=True),  # 짧은 변을 TARGET으로
    T.CenterCrop(TARGET),                                        # 정사각형 크롭
    # T.Resize((TARGET, TARGET), interpolation=IM.BICUBIC, antialias=True)  # <- 비율 무시 대안
])

# ------------ config ------------
DEVICE = torch.device('cuda:0')
JSON = 'mjhq_fid/prompts.json'
IMG_ROOT = Path('/dataset/mjhq30k')
OUT_DIR  = Path('mjhq_fid')
OUT_PATH = OUT_DIR / 'mjhq_30k_fid_stats.pt'
# --------------------------------

OUT_DIR.mkdir(parents=True, exist_ok=True)
net = FIDInception(device=DEVICE).eval()

import json
with open(JSON, 'r') as f:
    data = json.load(f)

feats = []
with torch.inference_mode():
    for key in tqdm(data, desc='Extracting Inception features'):
        p = IMG_ROOT / data[key]['category'] / f'{key}.jpg'
        with Image.open(p) as im:
            im = preproc(im.convert('RGB'))   # 🔹 전처리 추가 (PIL 유지)
        feats.append(net([im]).detach().cpu())  # (1, 2048)

feats = torch.cat(feats, dim=0).to(torch.float32)   # (N, 2048)
mu    = feats.mean(dim=0)                           # (2048,)
sigma = torch.cov(feats.T, correction=1)            # (2048, 2048) — unbiased

# ✅ Torch .pt로 통계만 저장
torch.save({'mu': mu.cpu(), 'sigma': sigma.cpu(), 'n': int(feats.shape[0])}, OUT_PATH)
print('done:', mu.shape, sigma.shape, '->', OUT_PATH)


Extracting Inception features: 100%|██████████| 30000/30000 [12:40<00:00, 39.44it/s]


done: torch.Size([2048]) torch.Size([2048, 2048]) -> mjhq_fid/mjhq_30k_fid_stats.pt


In [17]:
import json
with open('mjhq_fid/prompts.json', 'r') as f:
    json_data = json.load(f)
data = [json_data[key]['prompt'] for key in json_data]
data

['beautiful Jaguar decorated with huichol beads, in the jungle, plants everywhere, DMT colours, ultra realistic , cinematic lighting   v 5',
 'a hamster dressed in cia outfit',
 'Jesus the shepherd leading the sheep out of the fence in Palestine 2000 years ago',
 'beautiful pale pink baby fawn set among pastel pink flowers and soft wisteria, romantic specimen, pinkish white background, by Thomas kinkade Nadja Baxter Anne Stokes Nancy Noel ',
 'hyperrealistic portrait of a young woman, Belarus, over the shoulder, brown and long hair, white dress, tree of life, symbol of birth and fertility, shedding light around her, depicted with swan and horse, Nikon D850, 85mm lens, f1.8, intense contrast, vivid colors ',
 'A hyper detail painting in richard macneil style of a duck with her ducklings, walking through a field were there are cows grazing ',
 'A white horse in a storm of fire above the ocean ',
 'tiger cub playing with soccer ball ',
 'full body shot of a man with his head inside a croc